# Multi-Latent Attention (MLA) — Exercise

Companion to [Attention Part 2 § Multi-Latent Attention](https://tanulsingh.github.io/blog/attention-architectures#multi-latent-attention-deepseek-v2-2024).

MLA's idea: instead of *sharing* K, V across heads (like MQA/GQA), **compress** each token's representation to a tiny latent vector, and **reconstruct** per-head K, V from it on demand.

$$c_t = x_t \cdot W_{\text{down}} \quad (d_{\text{model}} \rightarrow d_c)$$
$$k_t^{(h)} = c_t \cdot W_{\text{up}}^{K,h}, \quad v_t^{(h)} = c_t \cdot W_{\text{up}}^{V,h}$$

During inference you only cache `c_t` (small), then reconstruct K, V on the fly for each head.

**Scope of this exercise:** we'll build the basic MLA without the absorbed-up-projection optimization or decoupled RoPE — those are CUDA-kernel territory. The core idea (compression + per-head reconstruction) is what you'll implement here.

## Setup

In [ ]:
# TODO: imports — torch, torch.nn as nn, math

## TODO 1 — `MultiLatentAttention`

Three projection groups:
1. **Q projection**: standard, like MHA — `W_q: d_model → n_heads * d_head`
2. **KV down-projection**: `W_kv_down: d_model → d_c` — compress to latent
3. **KV up-projections**: per-head, reconstruct K and V from latent
   - `W_k_up: d_c → n_heads * d_head` (combined for all K heads)
   - `W_v_up: d_c → n_heads * d_head` (combined for all V heads)
4. **Output projection**: `W_o: d_model → d_model`

In [ ]:
class MultiLatentAttention(nn.Module):
    def __init__(self, d_model: int, n_heads: int, d_c: int, causal: bool = False):
        """
        Args:
            d_model: model dimension
            n_heads: number of query/value heads
            d_c: latent dimension (compressed). Typical: d_c << d_model (e.g., 512 vs 5120)
            causal: whether to apply causal masking
        """
        super().__init__()
        # TODO 1a: store dimensions
        # TODO 1b: W_q: nn.Linear(d_model, n_heads * d_head)
        # TODO 1c: W_kv_down: nn.Linear(d_model, d_c)
        # TODO 1d: W_k_up: nn.Linear(d_c, n_heads * d_head)
        # TODO 1e: W_v_up: nn.Linear(d_c, n_heads * d_head)
        # TODO 1f: W_o: nn.Linear(d_model, d_model)
        pass

    def forward(self, x):
        # x: (batch, seq, d_model)
        # TODO 1g: Q = W_q(x) → reshape to (batch, n_heads, seq, d_head)
        # TODO 1h: c = W_kv_down(x)              # (batch, seq, d_c)  ← THIS is the cache
        # TODO 1i: K = W_k_up(c) → reshape to (batch, n_heads, seq, d_head)
        # TODO 1j: V = W_v_up(c) → reshape to (batch, n_heads, seq, d_head)
        # TODO 1k: standard scaled dot-product attention (with optional causal mask)
        # TODO 1l: reshape back and apply W_o
        pass

In [ ]:
# Sanity check:
#   - mla = MultiLatentAttention(d_model=128, n_heads=8, d_c=32)
#   - out = mla(torch.randn(2, 10, 128))
#   - out.shape == (2, 10, 128)
#   - The latent cache size per token is just d_c = 32 floats
#     vs full K+V cache of n_heads * d_head * 2 = 8 * 16 * 2 = 256 floats
#     → 8x compression

## TODO 2 — Verify per-head K, V are different

Unlike MQA/GQA where heads share the same K, V, in MLA each head reconstructs its own K, V from the shared latent. Verify this: extract K for two different heads from a forward pass, and confirm they're not identical.

In [ ]:
# TODO 2:
#   - Run forward, but capture intermediate K and V tensors
#   - K[batch, head=0, :, :] should NOT be close to K[batch, head=1, :, :]
#   - This is the key property: per-head diversity is preserved despite the shared latent

## Run the tests

In [ ]:
from tests import run_mla_tests
# run_mla_tests(MultiLatentAttention)